Ali Issabayev is responsible for this code

### RT-DETR Training on TACO Supercategory Dataset

#### 1. Environment Setup

In [5]:
!pip install ultralytics>=8.3.0 -q
!pip install torch torchvision -q
!pip install pandas matplotlib seaborn -q

#### 2. Dataset Configuration

Assuming the TACO dataset has already been converted to YOLO format with supercategories.

In [1]:
from pathlib import Path

DATASET_ROOT = Path("/home/default/Desktop/project/Dataset 4cats/taco_yolo_4cats")
YAML_CONFIG = DATASET_ROOT / "taco.yaml"

# Verify dataset structure
print("Dataset structure:")
for split in ["train", "val", "test"]:
    img_dir = DATASET_ROOT / f"images/{split}"
    lbl_dir = DATASET_ROOT / f"labels/{split}"
    if img_dir.exists() and lbl_dir.exists():
        n_imgs = len(list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png")))
        n_lbls = len(list(lbl_dir.glob("*.txt")))
        print(f"  {split}: {n_imgs} images, {n_lbls} labels")
    else:
        print(f"  {split}: NOT FOUND")

# Load and display class names
if YAML_CONFIG.exists():
    import yaml
    with open(YAML_CONFIG, 'r') as f:
        config = yaml.safe_load(f)
    print(f"\nNumber of classes: {len(config['names'])}")
    print(f"Classes: {list(config['names'].values())[:10]}...")

Dataset structure:
  train: 815 images, 1465 labels
  val: 230 images, 416 labels
  test: NOT FOUND

Number of classes: 4
Classes: ['plastic', 'glass', 'paper', 'unsorted']...


#### 3. Train RT-DETR

In [ ]:
from ultralytics import RTDETR
import os
os.environ["WANDB_DISABLED"] = "true"

model = RTDETR('rtdetr-l.pt')

# Training configuration
results = model.train(
    data=str(YAML_CONFIG),
    epochs=100,
    imgsz=640,
    batch=8,
    patience=30,
    device=0,
    project='runs/rtdetr',
    name='taco_4catrt',
    exist_ok=True,
    pretrained=True,
    
    # RT-DETR works better with AdamW
    optimizer='AdamW',
    lr0=0.0001,
    
    # Basic augmentations
    mosaic=0.5,
    copy_paste=0.3,   # Good for small objects
    mixup=0.1,
    
    # Enable validation
    val=True,
    save=True,
    plots=True,
)

print("\n" + "="*50)
print("Training completed!")
print(f"Best model: runs/rtdetr/taco_simple/weights/best.pt")
print("="*50)

New https://pypi.org/project/ultralytics/8.3.232 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.0 🚀 Python-3.10.18 torch-2.9.1+cu128 CUDA:0 (Tesla T4, 15948MiB)
engine/trainer: task=detect, mode=train, model=rtdetr-l.pt, data=/home/default/Desktop/project/Dataset 4cats/taco_yolo_4cats/taco.yaml, epochs=100, time=None, patience=30, batch=8, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=8, project=runs/rtdetr, name=taco_4catrt, exist_ok=True, pretrained=True, optimizer=AdamW, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, ret

100%|██████████| 6.25M/6.25M [00:00<00:00, 77.3MB/s]


AMP: checks passed ✅


train: Scanning /home/default/Desktop/project/Dataset 4cats/taco_yolo_4cats/labels/train.cache... 1465 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1465/1465 [00:00<?, ?it/s]
val: Scanning /home/default/Desktop/project/Dataset 4cats/taco_yolo_4cats/labels/val.cache... 416 images, 0 backgrounds, 0 corrupt: 100%|██████████| 416/416 [00:00<?, ?it/s]


Plotting labels to runs/rtdetr/taco_4catrt/labels.jpg... 
optimizer: AdamW(lr=0.0001, momentum=0.937) with parameter groups 143 weight(decay=0.0), 206 weight(decay=0.0005), 226 bias(decay=0.0)
MLflow: logging run_id(71452d9cdc2b4b77bbb104876ae366b4) to runs/mlflow
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri runs/mlflow'
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to runs/rtdetr/taco_4catrt
Starting training for 100 epochs...

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


...

  0%|          | 0/184 [00:00<?, ?it/s]/home/default/.local/lib/python3.10/site-packages/torch/autograd/graph.py:841: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:148.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
     68/100       7.4G      0.557     0.6638     0.1481          5        640: 100%|██████████| 184/184 [02:00<00:00,  1.53it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 26/26 [00:24<00:00,  1.07it/s]

                   all        416       1343      0.671       0.35       0.34      0.265



      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


  0%|          | 0/184 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 48.00 MiB. GPU 0 has a total capacity of 15.57 GiB of which 17.44 MiB is free. Including non-PyTorch memory, this process has 5.83 GiB memory in use. Process 49276 has 9.72 GiB memory in use. Of the allocated memory 5.57 GiB is allocated by PyTorch, and 64.59 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)